In [ ]:

import pandas as pd
import numpy as np

# Reproducible dataset
rng = np.random.default_rng(42)

# 1. CATEGORIES
categories = pd.DataFrame({
    "category_id": range(1, 21),
    "category_name": [
        "Beverages", "Snacks", "Dairy", "Bakery", "Frozen Food",
        "Personal Care", "Cleaning", "Health", "Pet Supplies",
        "Office Supplies", "Electronics", "Home & Kitchen",
        "Clothing", "Footwear", "Sports", "Toys",
        "Automotive", "Garden", "Baby Products", "Stationery"
    ]
})

# 2. SUPPLIERS
n_suppliers = 300

supplier_cities = [
    "Jakarta", "Bandung", "Surabaya", "Semarang", "Yogyakarta",
    "Medan", "Makassar", "Bekasi", "Tangerang", "Solo"
]

suppliers = pd.DataFrame({
    "supplier_id": range(1, n_suppliers + 1),
    "supplier_name": [
        f"Supplier {i:03d}" for i in range(1, n_suppliers + 1)
    ],
    "city": rng.choice(supplier_cities, n_suppliers),
    "supplier_tier": rng.choice(
        ["Standard", "Preferred", "Strategic"],
        n_suppliers,
        p=[0.60, 0.28, 0.12]
    ),
    "is_active": rng.choice(
        [True, False], n_suppliers, p=[0.95, 0.05]
    )
})

# 3. PRODUCTS
n_products = 2000

product_category = rng.integers(1, 21, n_products)
unit_cost = np.round(
    np.exp(rng.uniform(np.log(5000), np.log(2_000_000), n_products)),
    2
)
markup = rng.uniform(1.15, 1.80, n_products)
selling_price = np.round(unit_cost * markup, 2)

products = pd.DataFrame({
    "product_id": range(1, n_products + 1),
    "product_name": [
        f"Product {i:04d}" for i in range(1, n_products + 1)
    ],
    "category_id": product_category,
    "supplier_id": rng.integers(1, n_suppliers + 1, n_products),
    "unit_cost": unit_cost,
    "selling_price": selling_price,
    "reorder_point": rng.integers(20, 101, n_products),
    "safety_stock": rng.integers(10, 51, n_products),
    "is_active": rng.choice(
        [True, False], n_products, p=[0.97, 0.03]
    )
})

# 4. WAREHOUSES
warehouse_cities = [
    "Jakarta", "Bandung", "Surabaya", "Semarang", "Yogyakarta",
    "Medan", "Makassar", "Bekasi", "Tangerang", "Solo"
]

warehouses = pd.DataFrame({
    "warehouse_id": range(1, 11),
    "warehouse_name": [
        f"Warehouse {i:02d}" for i in range(1, 11)
    ],
    "city": warehouse_cities,
    "warehouse_capacity": rng.integers(50_000, 150_001, 10),
    "is_active": True
})

# Export master tables
master_tables = {
    "categories": categories,
    "suppliers": suppliers,
    "products": products,
    "warehouses": warehouses
}

for name, df in master_tables.items():
    df.to_csv(f"{name}.csv", index=False)
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")

# Basic validation
assert categories["category_id"].is_unique
assert suppliers["supplier_id"].is_unique
assert products["product_id"].is_unique
assert warehouses["warehouse_id"].is_unique

assert products["category_id"].isin(categories["category_id"]).all()
assert products["supplier_id"].isin(suppliers["supplier_id"]).all()
assert (products["unit_cost"] > 0).all()
assert (products["selling_price"] > products["unit_cost"]).all()

print("\nSemua validasi master data berhasil!")


categories: 20 rows, 2 columns
suppliers: 300 rows, 5 columns
products: 2,000 rows, 9 columns
warehouses: 10 rows, 5 columns

Semua validasi master data berhasil!


In [ ]:
display(categories.head())
display(suppliers.head())
display(products.head())
display(warehouses.head())

,category_id,category_name
0,1,Beverages
1,2,Snacks
2,3,Dairy
3,4,Bakery
4,5,Frozen Food


,supplier_id,supplier_name,city,supplier_tier,is_active
0,1,Supplier 001,Jakarta,Preferred,True
1,2,Supplier 002,Bekasi,Strategic,True
2,3,Supplier 003,Makassar,Standard,True
3,4,Supplier 004,Yogyakarta,Standard,True
4,5,Supplier 005,Yogyakarta,Standard,True


,product_id,product_name,category_id,supplier_id,unit_cost,selling_price,reorder_point,safety_stock,is_active
0,1,Product 0001,11,70,231969.70,279560.14,35,28,True
1,2,Product 0002,6,12,16725.38,24795.30,21,45,True
2,3,Product 0003,5,228,1022708.83,1475747.81,55,42,True
3,4,Product 0004,7,115,647438.28,1060732.38,41,20,True
4,5,Product 0005,12,88,40602.75,58954.86,22,17,True


,warehouse_id,warehouse_name,city,warehouse_capacity,is_active
0,1,Warehouse 01,Jakarta,54316,True
1,2,Warehouse 02,Bandung,81290,True
2,3,Warehouse 03,Surabaya,74101,True
3,4,Warehouse 04,Semarang,76520,True
4,5,Warehouse 05,Yogyakarta,149848,True


In [ ]:

# HISTORI PENJUALAN — 12 BULAN

import pandas as pd
import numpy as np

# Periode data: Januari–Desember 2025
n_orders = 60_000
start_date = pd.Timestamp("2025-01-01")
end_date = pd.Timestamp("2025-12-31")

# Buat tanggal pesanan secara acak
date_range = (end_date - start_date).days + 1
order_dates = start_date + pd.to_timedelta(
    rng.integers(0, date_range, n_orders), unit="D"
)

# Header pesanan
sales_orders = pd.DataFrame({
    "sales_order_id": range(1, n_orders + 1),
    "order_date": order_dates,
    "warehouse_id": rng.integers(1, 11, n_orders),
    "order_status": rng.choice(
        ["Completed", "Cancelled", "Returned"],
        n_orders,
        p=[0.92, 0.05, 0.03]
    )
})

# Detail pesanan: 1–5 baris produk per pesanan
items_per_order = rng.integers(1, 6, n_orders)
order_ids = np.repeat(sales_orders["sales_order_id"], items_per_order)
n_items = len(order_ids)

# Satu produk per baris detail; produk boleh berulang pada order berbeda
product_ids = rng.choice(
    products["product_id"].to_numpy(),
    size=n_items,
    replace=True
)

# Ambil harga dan biaya produk
product_lookup = products.set_index("product_id")

sales_order_items = pd.DataFrame({
    "sales_item_id": range(1, n_items + 1),
    "sales_order_id": order_ids,
    "product_id": product_ids
})

sales_order_items["unit_selling_price"] = (
    sales_order_items["product_id"]
    .map(product_lookup["selling_price"])
)

sales_order_items["unit_cost_at_sale"] = (
    sales_order_items["product_id"]
    .map(product_lookup["unit_cost"])
)

sales_order_items["quantity"] = rng.integers(1, 11, n_items)

# Simpan hasil
sales_orders.to_csv("sales_orders.csv", index=False)
sales_order_items.to_csv("sales_order_items.csv", index=False)

# Validasi
assert sales_orders["sales_order_id"].is_unique
assert sales_order_items["sales_item_id"].is_unique
assert sales_order_items["sales_order_id"].isin(
    sales_orders["sales_order_id"]
).all()
assert sales_order_items["product_id"].isin(
    products["product_id"]
).all()
assert sales_order_items["quantity"].gt(0).all()

print("Sales orders:", f"{len(sales_orders):,}", "baris")
print("Sales order items:", f"{len(sales_order_items):,}", "baris")
print("Rentang tanggal:", sales_orders["order_date"].min(),
      "sampai", sales_orders["order_date"].max())
print("Validasi penjualan berhasil!")


Sales orders: 60,000 baris
Sales order items: 180,076 baris
Rentang tanggal: 2025-01-01 00:00:00 sampai 2025-12-31 00:00:00
Validasi penjualan berhasil!


In [ ]:
display(sales_orders.head())
display(sales_order_items.head())

print("\nDistribusi status:")
display(sales_orders["order_status"].value_counts())

,sales_order_id,order_date,warehouse_id,order_status
0,1,2025-07-07,9,Completed
1,2,2025-06-20,5,Completed
2,3,2025-07-21,6,Completed
3,4,2025-11-03,2,Completed
4,5,2025-09-20,4,Completed


,sales_item_id,sales_order_id,product_id,unit_selling_price,unit_cost_at_sale,quantity
0,1,1,1366,387440.90,221122.56,6
0,2,1,1811,621266.54,354172.73,8
0,3,1,1938,1918855.23,1502709.61,2
1,4,2,1749,61027.18,52370.66,10
1,5,2,790,147132.91,106005.93,7



Distribusi status:


,count
order_status,
Completed,55153
Cancelled,3027
Returned,1820


In [ ]:

# HISTORI PEMBELIAN — 12 BULAN

n_purchase_orders = 8_000

# Tanggal pemesanan selama 2025
purchase_dates = start_date + pd.to_timedelta(
    rng.integers(0, (end_date - start_date).days + 1, n_purchase_orders),
    unit="D"
)

purchase_orders = pd.DataFrame({
    "purchase_order_id": range(1, n_purchase_orders + 1),
    "supplier_id": rng.integers(1, len(suppliers) + 1, n_purchase_orders),
    "warehouse_id": rng.integers(1, len(warehouses) + 1, n_purchase_orders),
    "order_date": purchase_dates
})

# Simulasi lead time supplier: 2–14 hari
lead_time_days = rng.integers(2, 15, n_purchase_orders)

purchase_orders["expected_delivery_date"] = (
    purchase_orders["order_date"] + pd.to_timedelta(lead_time_days, unit="D")
)

# Status pembelian
purchase_orders["purchase_status"] = rng.choice(
    ["Received", "Partial", "Pending", "Cancelled"],
    n_purchase_orders,
    p=[0.78, 0.10, 0.08, 0.04]
)

# Sebagian pengiriman tepat waktu, sebagian terlambat
delivery_delay = rng.integers(-2, 9, n_purchase_orders)

purchase_orders["actual_delivery_date"] = (
    purchase_orders["expected_delivery_date"]
    + pd.to_timedelta(delivery_delay, unit="D")
)

# Pesanan belum diterima atau dibatalkan tidak memiliki tanggal aktual
purchase_orders.loc[
    purchase_orders["purchase_status"].isin(["Pending", "Cancelled"]),
    "actual_delivery_date"
] = pd.NaT

# Jangan membuat tanggal penerimaan di masa depan untuk data 2025
future_received = (
    purchase_orders["actual_delivery_date"] > end_date
) & purchase_orders["purchase_status"].isin(["Received", "Partial"])

purchase_orders.loc[
    future_received, "actual_delivery_date"
] = pd.NaT

# Detail pembelian: 1–5 produk per purchase order
items_per_po = rng.integers(1, 6, n_purchase_orders)
po_ids = np.repeat(
    purchase_orders["purchase_order_id"].to_numpy(),
    items_per_po
)
n_po_items = len(po_ids)

po_product_ids = rng.choice(
    products["product_id"].to_numpy(),
    size=n_po_items,
    replace=True
)

purchase_order_items = pd.DataFrame({
    "purchase_item_id": range(1, n_po_items + 1),
    "purchase_order_id": po_ids,
    "product_id": po_product_ids,
    "ordered_qty": rng.integers(10, 201, n_po_items)
})

# Kuantitas diterima berdasarkan status header
status_map = purchase_orders.set_index(
    "purchase_order_id"
)["purchase_status"]

item_status = purchase_order_items["purchase_order_id"].map(status_map)

received_ratio = rng.uniform(0.80, 1.00, n_po_items)

purchase_order_items["received_qty"] = np.select(
    [
        item_status.eq("Received"),
        item_status.eq("Partial"),
        item_status.eq("Pending"),
        item_status.eq("Cancelled")
    ],
    [
        purchase_order_items["ordered_qty"],
        np.floor(
            purchase_order_items["ordered_qty"] * received_ratio
        ).astype(int),
        0,
        0
    ],
    default=0
)

product_lookup = products.set_index("product_id")

purchase_order_items["unit_purchase_cost"] = (
    purchase_order_items["product_id"]
    .map(product_lookup["unit_cost"])
)

# Simpan CSV
purchase_orders.to_csv("purchase_orders.csv", index=False)
purchase_order_items.to_csv("purchase_order_items.csv", index=False)

# Validasi dasar
assert purchase_orders["purchase_order_id"].is_unique
assert purchase_order_items["purchase_item_id"].is_unique
assert purchase_order_items["purchase_order_id"].isin(
    purchase_orders["purchase_order_id"]
).all()
assert (purchase_order_items["received_qty"] >= 0).all()
assert (
    purchase_order_items["received_qty"]
    <= purchase_order_items["ordered_qty"]
).all()

print("Purchase orders:", f"{len(purchase_orders):,}", "baris")
print("Purchase order items:", f"{len(purchase_order_items):,}", "baris")
print("\nDistribusi status:")
print(purchase_orders["purchase_status"].value_counts())
print("\nValidasi pembelian berhasil!")


Purchase orders: 8,000 baris
Purchase order items: 24,086 baris

Distribusi status:
purchase_status
Received     6263
Partial       799
Pending       603
Cancelled     335
Name: count, dtype: int64

Validasi pembelian berhasil!


In [ ]:

# AUDIT TANGGAL PENERIMAAN PEMBELIAN

print("Status pembelian:")
print(purchase_orders["purchase_status"].value_counts(dropna=False))

print("\nTanggal aktual kosong per status:")
print(
    purchase_orders.groupby("purchase_status")[
        "actual_delivery_date"
    ].apply(lambda s: s.isna().sum())
)

print("\nTanggal aktual sebelum tanggal pemesanan:")
print(
    (
        purchase_orders["actual_delivery_date"].notna()
        & (
            purchase_orders["actual_delivery_date"]
            < purchase_orders["order_date"]
        )
    ).sum()
)

print("\nTanggal aktual setelah 2025-12-31:")
print(
    (
        purchase_orders["actual_delivery_date"]
        > pd.Timestamp("2025-12-31")
    ).sum()
)


Status pembelian:
purchase_status
Received     6263
Partial       799
Pending       603
Cancelled     335
Name: count, dtype: int64

Tanggal aktual kosong per status:
purchase_status
Cancelled    335
Partial       18
Pending      603
Received     191
Name: actual_delivery_date, dtype: int64

Tanggal aktual sebelum tanggal pemesanan:
0

Tanggal aktual setelah 2025-12-31:
0


In [ ]:

# Perbaiki tanggal aktual yang kosong pada pesanan Received/Partial
# Gunakan tanggal target jika masih dalam periode 2025.
# Jika tanggal target melewati 2025, gunakan 31 Desember 2025.

mask_received_missing = (
    purchase_orders["purchase_status"].isin(["Received", "Partial"])
    & purchase_orders["actual_delivery_date"].isna()
)

purchase_orders.loc[mask_received_missing, "actual_delivery_date"] = (
    purchase_orders.loc[mask_received_missing, "expected_delivery_date"]
    .clip(upper=pd.Timestamp("2025-12-31"))
)

# Validasi ulang
assert purchase_orders.loc[
    purchase_orders["purchase_status"].isin(["Received", "Partial"]),
    "actual_delivery_date"
].notna().all()

assert (
    purchase_orders["actual_delivery_date"].isna()
    == purchase_orders["purchase_status"].isin(["Pending", "Cancelled"])
).all()

assert (
    purchase_orders["actual_delivery_date"].isna()
    | (
        purchase_orders["actual_delivery_date"]
        >= purchase_orders["order_date"]
    )
).all()

print("Tanggal penerimaan sudah konsisten.")
print(
    purchase_orders.groupby("purchase_status")[
        "actual_delivery_date"
    ].apply(lambda s: s.isna().sum())
)

# Simpan ulang
purchase_orders.to_csv("purchase_orders.csv", index=False)


Tanggal penerimaan sudah konsisten.
purchase_status
Cancelled    335
Partial        0
Pending      603
Received       0
Name: actual_delivery_date, dtype: int64


In [ ]:

# LANGKAH 3 — GENERATE INVENTORY SNAPSHOTS

import pandas as pd
import numpy as np

# Pastikan tanggal bertipe datetime
sales_orders["order_date"] = pd.to_datetime(
    sales_orders["order_date"]
)

purchase_orders["order_date"] = pd.to_datetime(
    purchase_orders["order_date"]
)
purchase_orders["actual_delivery_date"] = pd.to_datetime(
    purchase_orders["actual_delivery_date"]
)

months = pd.date_range("2025-01-01", "2025-12-01", freq="MS")

# --------------------------------------------------
# 1. AGREGASI PENJUALAN PER BULAN, PRODUK, GUDANG
# --------------------------------------------------

sales_detail = sales_order_items.merge(
    sales_orders[
        ["sales_order_id", "order_date", "warehouse_id", "order_status"]
    ],
    on="sales_order_id",
    how="left",
    validate="many_to_one"
)

sales_detail["snapshot_month"] = (
    sales_detail["order_date"].dt.to_period("M").dt.to_timestamp()
)

# Hanya pesanan Completed yang mengurangi stok
sales_monthly = (
    sales_detail[sales_detail["order_status"] == "Completed"]
    .groupby(
        ["snapshot_month", "product_id", "warehouse_id"],
        as_index=False
    )["quantity"]
    .sum()
    .rename(columns={"quantity": "sold_qty"})
)

# Retur menambah stok kembali
returns_monthly = (
    sales_detail[sales_detail["order_status"] == "Returned"]
    .groupby(
        ["snapshot_month", "product_id", "warehouse_id"],
        as_index=False
    )["quantity"]
    .sum()
    .rename(columns={"quantity": "adjustment_qty"})
)

# --------------------------------------------------
# 2. AGREGASI BARANG DITERIMA PER BULAN
# --------------------------------------------------

purchase_detail = purchase_order_items.merge(
    purchase_orders[
        [
            "purchase_order_id",
            "warehouse_id",
            "actual_delivery_date",
            "purchase_status"
        ]
    ],
    on="purchase_order_id",
    how="left",
    validate="many_to_one"
)

valid_receipts = purchase_detail[
    purchase_detail["purchase_status"].isin(["Received", "Partial"])
    & purchase_detail["actual_delivery_date"].notna()
].copy()

valid_receipts["snapshot_month"] = (
    valid_receipts["actual_delivery_date"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

received_monthly = (
    valid_receipts.groupby(
        ["snapshot_month", "product_id", "warehouse_id"],
        as_index=False
    )["received_qty"]
    .sum()
)

# --------------------------------------------------
# 3. BENTUK KOMBINASI 2.000 PRODUK x 10 GUDANG x 12 BULAN
# --------------------------------------------------

grid = pd.MultiIndex.from_product(
    [
        months,
        products["product_id"].to_numpy(),
        warehouses["warehouse_id"].to_numpy()
    ],
    names=["snapshot_month", "product_id", "warehouse_id"]
).to_frame(index=False)

grid = grid.merge(
    received_monthly,
    on=["snapshot_month", "product_id", "warehouse_id"],
    how="left",
    validate="one_to_one"
)

grid = grid.merge(
    sales_monthly,
    on=["snapshot_month", "product_id", "warehouse_id"],
    how="left",
    validate="one_to_one"
)

grid = grid.merge(
    returns_monthly,
    on=["snapshot_month", "product_id", "warehouse_id"],
    how="left",
    validate="one_to_one"
)

for col in ["received_qty", "sold_qty", "adjustment_qty"]:
    grid[col] = grid[col].fillna(0).astype("int64")

# Biaya valuasi stok
grid["unit_cost"] = grid["product_id"].map(
    products.set_index("product_id")["unit_cost"]
)

grid = grid.sort_values(
    ["product_id", "warehouse_id", "snapshot_month"]
).reset_index(drop=True)

# --------------------------------------------------
# 4. TENTUKAN STOK AWAL JANUARI AGAR TIDAK NEGATIF
# --------------------------------------------------

# Perubahan stok bulanan sebelum stok awal diperhitungkan
grid["net_movement"] = (
    grid["received_qty"]
    - grid["sold_qty"]
    + grid["adjustment_qty"]
)

# Akumulasi per produk-gudang
grid["cumulative_movement"] = (
    grid.groupby(["product_id", "warehouse_id"])["net_movement"]
    .cumsum()
)

# Cari penurunan kumulatif terendah selama setahun
min_cumulative = (
    grid.groupby(["product_id", "warehouse_id"])[
        "cumulative_movement"
    ]
    .min()
)

# Buffer stok awal acak yang positif
initial_buffer = pd.Series(
    rng.integers(10, 51, len(min_cumulative)),
    index=min_cumulative.index
)

initial_stock = (
    initial_buffer + (-min_cumulative).clip(lower=0)
).astype("int64")
# Rumus di atas menjaga stok awal cukup untuk seluruh pergerakan
# dan menyisakan buffer positif pada akhir setiap bulan.
grid["initial_stock"] = [
    initial_stock.loc[(p, w)]
    for p, w in zip(grid["product_id"], grid["warehouse_id"])
]

# --------------------------------------------------
# 5. HITUNG STOK AWAL DAN AKHIR BULAN
# --------------------------------------------------

grid["beginning_stock"] = (
    grid["initial_stock"]
    + grid["cumulative_movement"]
    - grid["net_movement"]
)

grid["ending_stock"] = (
    grid["beginning_stock"]
    + grid["received_qty"]
    - grid["sold_qty"]
    + grid["adjustment_qty"]
)

# Susun kolom sesuai skema database
inventory_snapshots = grid[
    [
        "snapshot_month",
        "product_id",
        "warehouse_id",
        "beginning_stock",
        "received_qty",
        "sold_qty",
        "adjustment_qty",
        "ending_stock",
        "unit_cost"
    ]
].copy()

inventory_snapshots.insert(
    0, "snapshot_id", range(1, len(inventory_snapshots) + 1)
)

# --------------------------------------------------
# 6. VALIDASI
# --------------------------------------------------

assert len(inventory_snapshots) == 240_000

assert inventory_snapshots["snapshot_id"].is_unique

assert not inventory_snapshots.duplicated(
    ["snapshot_month", "product_id", "warehouse_id"]
).any()

assert (
    inventory_snapshots["ending_stock"]
    ==
    inventory_snapshots["beginning_stock"]
    + inventory_snapshots["received_qty"]
    - inventory_snapshots["sold_qty"]
    + inventory_snapshots["adjustment_qty"]
).all()

assert (inventory_snapshots["beginning_stock"] >= 0).all()
assert (inventory_snapshots["ending_stock"] >= 0).all()

# Stok akhir bulan harus sama dengan stok awal bulan berikutnya
check = inventory_snapshots.sort_values(
    ["product_id", "warehouse_id", "snapshot_month"]
).copy()

check["next_beginning_stock"] = (
    check.groupby(["product_id", "warehouse_id"])["beginning_stock"]
    .shift(-1)
)

has_next_month = check["next_beginning_stock"].notna()

assert (
    check.loc[has_next_month, "ending_stock"].to_numpy()
    ==
    check.loc[has_next_month, "next_beginning_stock"].to_numpy()
).all()

# --------------------------------------------------
# 7. EXPORT CSV
# --------------------------------------------------

inventory_snapshots.to_csv(
    "inventory_snapshots.csv",
    index=False
)

print("Inventory snapshots:", f"{len(inventory_snapshots):,}", "baris")
print("Periode:", months.min().date(), "sampai", months.max().date())
print("Stok akhir minimum:", inventory_snapshots["ending_stock"].min())
print("Semua validasi inventory berhasil!")


Inventory snapshots: 240,000 baris
Periode: 2025-01-01 sampai 2025-12-01
Stok akhir minimum: 10
Semua validasi inventory berhasil!


In [ ]:

# KOREKSI INVENTORY SNAPSHOTS
# Pesanan Returned tidak memengaruhi stok pada versi dataset ini.

import pandas as pd
import numpy as np

# 1. Pastikan kolom pergerakan stok tersedia
grid["received_qty"] = grid["received_qty"].fillna(0).astype(int)
grid["sold_qty"] = grid["sold_qty"].fillna(0).astype(int)

# 2. Hilangkan penambahan stok dari pesanan Returned
grid["adjustment_qty"] = 0

# 3. Hitung pergerakan stok bersih per bulan
grid = grid.sort_values(
    ["product_id", "warehouse_id", "snapshot_month"]
).reset_index(drop=True)

grid["net_movement"] = (
    grid["received_qty"]
    - grid["sold_qty"]
    + grid["adjustment_qty"]
)

group_cols = ["product_id", "warehouse_id"]

grid["cumulative_movement"] = (
    grid.groupby(group_cols)["net_movement"].cumsum()
)

# 4. Tentukan stok awal agar stok tidak pernah negatif
min_cumulative = (
    grid.groupby(group_cols)["cumulative_movement"]
    .transform("min")
)

# Buffer stok awal 20–100 unit per kombinasi produk-gudang
rng = np.random.default_rng(42)

initial_buffer = (
    grid.groupby(group_cols).ngroup()
    .map(dict(enumerate(
        rng.integers(20, 101, size=grid.groupby(group_cols).ngroups)
    )))
)

grid["initial_stock"] = (
    initial_buffer + (-min_cumulative).clip(lower=0)
)

# 5. Hitung stok awal dan akhir setiap bulan
grid["beginning_stock"] = (
    grid["initial_stock"]
    + grid["cumulative_movement"]
    - grid["net_movement"]
)

grid["ending_stock"] = (
    grid["beginning_stock"]
    + grid["received_qty"]
    - grid["sold_qty"]
    + grid["adjustment_qty"]
)

# 6. Bentuk dataset final
inventory_snapshots = grid[
    [
        "snapshot_month",
        "product_id",
        "warehouse_id",
        "beginning_stock",
        "received_qty",
        "sold_qty",
        "adjustment_qty",
        "ending_stock",
        "unit_cost",
    ]
].copy()

inventory_snapshots.insert(
    0, "snapshot_id", range(1, len(inventory_snapshots) + 1)
)

# 7. Validasi bisnis
assert len(inventory_snapshots) == 240_000
assert not inventory_snapshots.duplicated(
    ["snapshot_month", "product_id", "warehouse_id"]
).any()

assert (
    inventory_snapshots["ending_stock"]
    == inventory_snapshots["beginning_stock"]
    + inventory_snapshots["received_qty"]
    - inventory_snapshots["sold_qty"]
    + inventory_snapshots["adjustment_qty"]
).all()

assert (inventory_snapshots["beginning_stock"] >= 0).all()
assert (inventory_snapshots["ending_stock"] >= 0).all()
assert (inventory_snapshots["adjustment_qty"] == 0).all()

# Stok akhir bulan ini = stok awal bulan berikutnya
check = inventory_snapshots.sort_values(
    ["product_id", "warehouse_id", "snapshot_month"]
).copy()

next_beginning = (
    check.groupby(group_cols)["beginning_stock"].shift(-1)
)

has_next_month = check.groupby(group_cols).cumcount() < 11

assert (
    check.loc[has_next_month, "ending_stock"].to_numpy()
    == next_beginning[has_next_month].to_numpy()
).all()

# 8. Simpan ulang CSV yang sudah diperbaiki
inventory_snapshots.to_csv(
    "inventory_snapshots.csv", index=False
)

print("VALIDASI LOLOS")
print("Jumlah baris:", len(inventory_snapshots))
print("Adjustment selain nol:",
      (inventory_snapshots["adjustment_qty"] != 0).sum())
print("Stok akhir minimum:",
      inventory_snapshots["ending_stock"].min())
print("File berhasil diperbarui: inventory_snapshots.csv")


VALIDASI LOLOS
Jumlah baris: 240000
Adjustment selain nol: 0
Stok akhir minimum: 20
File berhasil diperbarui: inventory_snapshots.csv


In [ ]:
print(inventory_snapshots.head())
print(inventory_snapshots.info())

print("\nJumlah baris per tabel:")
for name, df in {
    "categories": categories,
    "suppliers": suppliers,
    "products": products,
    "warehouses": warehouses,
    "sales_orders": sales_orders,
    "sales_order_items": sales_order_items,
    "purchase_orders": purchase_orders,
    "purchase_order_items": purchase_order_items,
    "inventory_snapshots": inventory_snapshots
}.items():
    print(f"{name}: {len(df):,}")

   snapshot_id snapshot_month  product_id  warehouse_id  beginning_stock  \
0            1     2025-01-01           1             1               27   
1            2     2025-02-01           1             1               27   
2            3     2025-03-01           1             1               27   
3            4     2025-04-01           1             1               27   
4            5     2025-05-01           1             1               27   

   received_qty  sold_qty  adjustment_qty  ending_stock  unit_cost  
0             0         0               0            27   231969.7  
1             0         0               0            27   231969.7  
2             0         0               0            27   231969.7  
3             0         0               0            27   231969.7  
4           142         5               0           164   231969.7  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240000 entries, 0 to 239999
Data columns (total 10 columns):
 #   Column       

In [ ]:
import pandas as pd
from pathlib import Path

for file in Path(".").glob("*.csv"):
    try:
        df = pd.read_csv(file)

        if "is_active" in df.columns:
            df["is_active"] = df["is_active"].replace({
                True: 1,
                False: 0,
                "True": 1,
                "False": 0
            })

            df.to_csv(file, index=False)
            print(f"Berhasil diperbaiki: {file.name}")

    except Exception as e:
        print(f"Gagal memproses {file.name}: {e}")

Berhasil diperbaiki: warehouses.csv
Berhasil diperbaiki: suppliers.csv
Berhasil diperbaiki: products.csv


/tmp/ipykernel_2033/3108696590.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["is_active"] = df["is_active"].replace({
/tmp/ipykernel_2033/3108696590.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["is_active"] = df["is_active"].replace({
/tmp/ipykernel_2033/3108696590.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.